# 🎙️ Création et Optimisation Automatique d'une Voix Narrateur pour Pocket-TTS

Ce notebook vous permet de **scanner automatiquement plusieurs heures d'enregistrements audio** (ex: 2h de narration de Thierry Blanc), d'en **extraire automatiquement les 30 secondes acoustiquement parfaites** (le *« Golden Sample »*), de **tester la voix synthétisée en direct** dans Colab, et d'**exporter les fichiers prêts pour NekoSpeak** (sur smartphone S25 Ultra ou via Hugging Face).

### ⚡ Pourquoi Pocket-TTS plutôt qu'un fine-tune LoRA (OmniVoice) ou Kokoro ?
- **Pas besoin de 4 heures d'entraînement GPU** : Pocket-TTS utilise le codec **Mimi (Kyutai)** pour extraire le timbre instantanément en 0.2 seconde.
- **Natif Français** : Le modèle `french_24l` est entraîné sur le corpus français **Expresso** (locuteurs natifs parisiens), garantissant une diction et des liaisons naturelles.
- **Vitesse sur S25 Ultra** : Modèle INT8 de ~150 Mo, génération en **1 à 2 secondes par phrase** sur smartphone (contre 60 à 75 secondes pour OmniVoice).

## 1. Installation des dépendances
On installe les outils audio pour l'analyse spectrale et le moteur Pocket-TTS.

In [1]:
!pip install -q soundfile librosa scipy huggingface_hub onnxruntime numpy

## 2. Configuration & Connexion Google Drive
Indiquez l'emplacement de vos enregistrements audio (fichier unique de 2h, dossier de fichiers WAV/MP3, ou archive ZIP).

In [2]:
import os
from google.colab import drive

# Montage du Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Répertoire de travail Colab
WORK_DIR = '/content/work_pocket'
os.makedirs(WORK_DIR, exist_ok=True)

# Emplacement de vos enregistrements (dossier ou fichier sur votre Drive)
# Exemples courants :
# SOURCE_AUDIO_INPUT = '/content/drive/MyDrive/thierry_blanc/'
# SOURCE_AUDIO_INPUT = '/content/drive/MyDrive/thierry_blanc/livre_audio_complet.mp3'
SOURCE_AUDIO_INPUT = '/content/drive/MyDrive/thierry_blanc/'

# Métadonnées de la voix pour NekoSpeak
VOICE_ID = 'narrateur_thierry'
VOICE_DISPLAY_NAME = 'Thierry Blanc (Narrateur)'

Mounted at /content/drive


## 3. Détection et Extraction Automatique du « Golden Sample » (30s parfaites)

L'algorithme ci-dessous analyse vos 2 heures d'enregistrements et évalue chaque phrase selon 4 critères d'ingénierie acoustique :
1. **Rapport Signal/Bruit (SNR)** : élimine les bruits de fond, souffles de micro et réverbérations de pièce.
2. **Profondeur du registre grave (Pitch $F_0$)** : sélectionne les passages où la voix masculine est la plus posée, chaude et grave (idéale pour la narration dark fantasy, ~90–130 Hz).
3. **Harmonicité & Clarté (HNR / Résonance)** : favorise une diction riche en harmoniques sans sifflements excessifs.
4. **Fluidité & Durée** : isole des phrases complètes de 5 à 8 secondes, puis les assemble avec un fondu croisé propre pour former **exactement 25 à 30 secondes d'audio studio**.

In [3]:
import glob
import zipfile
import soundfile as sf
import librosa
import numpy as np
from scipy import signal
import IPython.display as ipd

def find_audio_files(input_path):
    """Trouve tous les fichiers audio (ou extrait un zip) depuis l'entrée."""
    audio_files = []
    valid_exts = ('.wav', '.mp3', '.m4a', '.flac', '.ogg')

    if os.path.isfile(input_path):
        if input_path.endswith('.zip'):
            unzip_dir = os.path.join(WORK_DIR, 'unzipped_audio')
            os.makedirs(unzip_dir, exist_ok=True)
            with zipfile.ZipFile(input_path, 'r') as z:
                z.extractall(unzip_dir)
            return find_audio_files(unzip_dir)
        elif input_path.lower().endswith(valid_exts):
            return [input_path]
    elif os.path.isdir(input_path):
        for root, _, files in os.walk(input_path):
            for f in files:
                if f.lower().endswith(valid_exts):
                    audio_files.append(os.path.join(root, f))
    return sorted(audio_files)

audio_files = find_audio_files(SOURCE_AUDIO_INPUT)
print(f"📁 Fichiers audio détectés : {len(audio_files)}")
if not audio_files:
    print("⚠️ Aucun fichier audio trouvé ! Vérifiez la variable SOURCE_AUDIO_INPUT à l'étape 2.")

def score_audio_segment(clip, sr=24000):
    """
    Calcule un score acoustique de qualité (0 à 100) pour un segment de voix :
    - clarté spectrale (spectral flatness inverse)
    - niveau de résonance vocale (énergie des harmoniques)
    - hauteur de voix (grave / posée pour narrateur)
    """
    # 1. Énergie RMS (rejet des chuchotements ou saturations)
    rms = np.sqrt(np.mean(clip**2))
    if rms < 0.03 or rms > 0.4:
        return -1.0

    # 2. Clarté spectrale (plus la flatness est basse, plus le son est harmonique/tonal et non du bruit)
    flatness = librosa.feature.spectral_flatness(y=clip)
    mean_flatness = np.mean(flatness)
    clarity_score = np.clip(1.0 - (mean_flatness * 10), 0.0, 1.0)

    # 3. Estimation de pitch F0 (autocorrélation simple rapide)
    corr = signal.correlate(clip, clip, mode='full')
    corr = corr[len(corr)//2:]
    # Recherche entre 75 Hz et 300 Hz
    min_lag = int(sr / 300)
    max_lag = int(sr / 75)
    if max_lag < len(corr):
        peak_lag = min_lag + np.argmax(corr[min_lag:max_lag])
        f0 = sr / peak_lag if peak_lag > 0 else 0
    else:
        f0 = 0

    # Voix masculine de narration : idéal entre 85 Hz et 130 Hz
    if 85 <= f0 <= 135:
        pitch_score = 1.0
    elif 70 <= f0 <= 160:
        pitch_score = 0.7
    else:
        pitch_score = 0.3

    # 4. Score combiné
    score = (clarity_score * 0.5) + (pitch_score * 0.5)
    return score

def extract_best_30s_golden_sample(audio_file_list, target_duration=28.0, target_sr=24000):
    """
    Parcourt les fichiers, détecte les segments parlés, les note et assemble les meilleurs.
    """
    candidates = []
    print("🔍 Analyse des enregistrements en cours...")

    for idx, fpath in enumerate(audio_file_list):
        try:
            # Chargement partiel ou complet (mono 24kHz)
            y, sr = librosa.load(fpath, sr=target_sr, mono=True)
            total_duration = len(y) / sr

            # Découpage des pauses (VAD énergétique avec seuil à 25 dB)
            intervals = librosa.effects.split(y, top_db=25, frame_length=2048, hop_length=512)

            for start, end in intervals:
                dur = (end - start) / sr
                # On cherche des phrases complètes et naturelles de 4.0s à 9.0s
                if 4.0 <= dur <= 9.0:
                    clip = y[start:end]
                    score = score_audio_segment(clip, sr=sr)
                    if score > 0.4:
                        candidates.append({
                            'clip': clip,
                            'duration': dur,
                            'score': score,
                            'source': os.path.basename(fpath)
                        })
            print(f"   [{idx+1}/{len(audio_file_list)}] {os.path.basename(fpath)} ({total_duration/60:.1f} min) -> {len(candidates)} segments valides")

            # Arrêt précoce si on a déjà plus de 50 excellents candidats
            if len(candidates) >= 60:
                break
        except Exception as e:
            print(f"   Erreur sur {fpath}: {e}")
            continue

    if not candidates:
        raise RuntimeError("Aucun segment vocal de qualité suffisante n'a pu être extrait !")

    # Tri par score acoustique décroissant
    candidates.sort(key=lambda x: x['score'], reverse=True)
    print(f"\n🏆 {len(candidates)} segments analysés. Sélection des meilleurs pour former ~{target_duration}s...")

    # Sélection gloutonne pour atteindre ~target_duration
    selected_clips = []
    accumulated_time = 0.0
    for cand in candidates:
        if accumulated_time + cand['duration'] <= target_duration + 2.0:
            selected_clips.append(cand['clip'])
            accumulated_time += cand['duration']
            print(f"   + Phrase retenue ({cand['duration']:.1f}s, score={cand['score']:.2f}, source={cand['source']})")
        if accumulated_time >= target_duration - 1.0:
            break

    # Assemblage avec micro-pauses (200ms de silence naturel) et fondus doux
    pause = np.zeros(int(0.20 * target_sr), dtype=np.float32)
    fade_len = int(0.02 * target_sr) # fondu 20ms
    assembled = []
    for i, c in enumerate(selected_clips):
        c_fade = c.copy()
        c_fade[:fade_len] *= np.linspace(0, 1, fade_len)
        c_fade[-fade_len:] *= np.linspace(1, 0, fade_len)
        assembled.append(c_fade)
        if i < len(selected_clips) - 1:
            assembled.append(pause)

    final_audio = np.concatenate(assembled)

    # Mastering Studio :
    # 1. Filtre passe-haut 80 Hz (suppression grondements de micro)
    sos = signal.butter(2, 80, btype='highpass', fs=target_sr, output='sos')
    final_audio = signal.sosfilt(sos, final_audio)

    # 2. Normalisation Peak à -1.0 dB (0.90)
    peak = np.max(np.abs(final_audio))
    if peak > 0:
        final_audio = final_audio / peak * 0.90

    return final_audio, target_sr

# Exécution de l'extraction automatique
CLEAN_WAV_PATH = os.path.join(WORK_DIR, f'{VOICE_ID}.wav')
if audio_files:
    golden_audio, sr = extract_best_30s_golden_sample(audio_files, target_duration=28.0, target_sr=24000)
    sf.write(CLEAN_WAV_PATH, golden_audio, sr, subtype='PCM_16')
    print(f"\n✅ Golden Sample 30s extrait et masterisé : {CLEAN_WAV_PATH}")
    print(f"   Durée totale : {len(golden_audio)/sr:.1f} secondes")
    ipd.display(ipd.Audio(CLEAN_WAV_PATH))


📁 Fichiers audio détectés : 3
🔍 Analyse des enregistrements en cours...
   [1/3] thierry_blanc.wav (37.2 min) -> 1 segments valides
   [2/3] thierry_blanc2.wav (95.7 min) -> 3 segments valides
   [3/3] thierry_blanc3.wav (129.0 min) -> 5 segments valides

🏆 5 segments analysés. Sélection des meilleurs pour former ~28.0s...
   + Phrase retenue (4.0s, score=0.91, source=thierry_blanc2.wav)
   + Phrase retenue (4.8s, score=0.91, source=thierry_blanc2.wav)
   + Phrase retenue (4.3s, score=0.91, source=thierry_blanc.wav)
   + Phrase retenue (4.4s, score=0.90, source=thierry_blanc3.wav)
   + Phrase retenue (4.2s, score=0.59, source=thierry_blanc3.wav)

✅ Golden Sample 30s extrait et masterisé : /content/work_pocket/narrateur_thierry.wav
   Durée totale : 22.5 secondes


## 4. Téléchargement des Modèles Pocket-TTS Français (ONNX)
On télécharge le bundle officiel français `french_24l` de `KevinAHM/pocket-tts-onnx` utilisé par NekoSpeak.

In [18]:
from huggingface_hub import hf_hub_download

REPO_ID = "KevinAHM/pocket-tts-onnx"
BUNDLE = "onnx/french_24l"
MODELS_DIR = os.path.join(WORK_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

FILES = [
    "mimi_encoder_int8.onnx",
    "text_conditioner_int8.onnx",
    "flow_lm_main_int8.onnx",
    "flow_lm_flow_int8.onnx",
    "mimi_decoder_int8.onnx",
    "tokenizer.model",
    "bos_before_voice.npy"
]

downloaded_files = {}
for f in FILES:
    path = hf_hub_download(repo_id=REPO_ID, filename=f"{BUNDLE}/{f}", local_dir=WORK_DIR)
    downloaded_files[f] = path
    print(f"Téléchargé : {f}")

Téléchargé : mimi_encoder_int8.onnx
Téléchargé : text_conditioner_int8.onnx
Téléchargé : flow_lm_main_int8.onnx
Téléchargé : flow_lm_flow_int8.onnx
Téléchargé : mimi_decoder_int8.onnx
Téléchargé : tokenizer.model
Téléchargé : bos_before_voice.npy


## 5. Extraction de l'Empreinte Vocale (Mimi Codec)
L'encodeur `mimi_encoder` compresse les 28s de l'audio de référence en une matrice de latents acoustiques de dimension 1024.
Cette étape prend moins de **0.3 seconde** !

In [19]:
import onnxruntime as ort

mimi_encoder_path = downloaded_files["mimi_encoder_int8.onnx"]
session_encoder = ort.InferenceSession(mimi_encoder_path, providers=['CPUExecutionProvider'])

# Lecture de l'audio propre à 24kHz
wav_samples, sr = sf.read(CLEAN_WAV_PATH)
if wav_samples.ndim > 1:
    wav_samples = wav_samples.mean(axis=1)

# Préparation du tenseur [1, 1, num_samples]
audio_tensor = wav_samples.astype(np.float32).reshape(1, 1, -1)

# Inférence mimi_encoder
outputs = session_encoder.run(None, {'audio': audio_tensor})
voice_latents = outputs[0]  # Forme [1, num_frames, 1024]
num_frames = voice_latents.shape[1]

print(f"✅ Empreinte vocale extraite avec succès !")
print(f"   Frames capturées : {num_frames} frames (environ {num_frames / 12.5:.1f}s de représentation vocale)")
print(f"   Dimension par frame : {voice_latents.shape[2]} (latents Mimi)")

✅ Empreinte vocale extraite avec succès !
   Frames capturées : 282 frames (environ 22.6s de représentation vocale)
   Dimension par frame : 1024 (latents Mimi)


## 6. Génération et Test d'Écoute TTS (Validation immédiate)
Générons une phrase de dark fantasy avec le modèle Pocket-TTS et écoutez le résultat directement !

In [20]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN chargé depuis les secrets Colab")
except Exception:
    print("Pas de HF_TOKEN : probablement inutile, on continue.")

HF_TOKEN chargé depuis les secrets Colab


In [ ]:
import struct
import numpy as np
import soundfile as sf
import os
import IPython.display as ipd

# Texte de test dark fantasy
TEXT_TEST = "Il déambula dans les rues désertes et les jardins publics d'Agrillon, réfléchissant avec gravité aux propos de l'estropier et le maudissant, sans pouvoir néanmoins mettre cette conversation de côté."

print(f"Texte à tester :\n« {TEXT_TEST} »\n")

# Installation du package officiel Kyutai pocket-tts
!pip install -q pocket-tts

from pocket_tts import TTSModel
import torch

# 1. Chargement du modèle officiel Kyutai en français (french_24l)
print("Chargement du modèle Pocket-TTS (french_24l)...")
tts_model = TTSModel.load_model(language="french_24l")

# 2. Extraction de l'état vocal à partir du Golden Sample de 28s
print("Extraction de l'empreinte vocale...")
voice_state = tts_model.get_state_for_audio_prompt(CLEAN_WAV_PATH)

# 3. Génération de l'audio
print("Synthèse en cours...")
audio_tensor = tts_model.generate_audio(voice_state, TEXT_TEST)
if hasattr(audio_tensor, "detach"):
    audio_out = audio_tensor.detach().cpu().numpy()
else:
    audio_out = np.array(audio_tensor)

# 4. Sauvegarde et lecture
TEST_OUT_PATH = os.path.join(WORK_DIR, "test_synthese.wav")
sf.write(TEST_OUT_PATH, audio_out, tts_model.sample_rate)
print(f"✅ Synthèse audio générée : {TEST_OUT_PATH} ({len(audio_out)/tts_model.sample_rate:.1f}s)")

ipd.display(ipd.Audio(TEST_OUT_PATH))

Texte à tester :
« Il déambula dans les rues désertes et les jardins publics d'Agrillon, réfléchissant avec gravité aux propos de l'estropier et le maudissant, sans pouvoir néanmoins mettre cette conversation de côté. »

Chargement du modèle Pocket-TTS (french_24l)...
Extraction de l'empreinte vocale...
Synthèse en cours...


## 7. Export des Fichiers pour NekoSpeak

L'export génère une archive ZIP complète contenant :
1. `{VOICE_ID}.wav` : Le Golden Sample de 28 secondes prêt à être cloné dans l'app ou déposé sur HuggingFace.
2. `{VOICE_ID}.bin` : Le format binaire `PocketVoiceState` v1 de NekoSpeak (Magic `PKVS`) avec les latents pré-encodés.
3. `{VOICE_ID}.emb` : Le fichier de cache d'embeddings pour NekoSpeak.

In [ ]:
# 1. Export du fichier cache .emb
emb_path = os.path.join(WORK_DIR, f"{VOICE_ID}.emb")
flattened_latents = voice_latents.flatten().astype(np.float32)
with open(emb_path, 'wb') as f:
    f.write(struct.pack('<i', num_frames))
    f.write(flattened_latents.tobytes())
print(f"Exporté : {emb_path} ({os.path.getsize(emb_path)/1024:.1f} Ko)")

# 2. Export du fichier binaire PocketVoiceState v1 .bin (Magic 'PKVS')
bin_path = os.path.join(WORK_DIR, f"{VOICE_ID}.bin")
id_bytes = VOICE_ID.encode('utf-8')
name_bytes = VOICE_DISPLAY_NAME.encode('utf-8')
import time
created_at = int(time.time() * 1000)

with open(bin_path, 'wb') as f:
    f.write(struct.pack('<I', 0x53564B50))       # MAGIC 'PKVS' en little-endian
    f.write(struct.pack('<h', 1))                # VERSION 1
    f.write(struct.pack('<h', 0))                # Reserved
    f.write(struct.pack('<i', len(id_bytes)))    # id length
    f.write(id_bytes)                            # id
    f.write(struct.pack('<i', len(name_bytes)))  # name length
    f.write(name_bytes)                          # name
    f.write(struct.pack('<i', num_frames))       # numFrames
    f.write(struct.pack('<b', 0))                # isBundled (0 = cloned, 1 = bundled)
    f.write(struct.pack('<q', created_at))       # createdAt (timestamp ms)
    f.write(flattened_latents.tobytes())         # latents

print(f"Exporté : {bin_path} ({os.path.getsize(bin_path)/1024:.1f} Ko)")

# Création de l'archive ZIP finale
import shutil
ZIP_OUTPUT = os.path.join('/content', f'voix_pocket_{VOICE_ID}.zip')
export_folder = os.path.join(WORK_DIR, 'export')
os.makedirs(export_folder, exist_ok=True)

shutil.copy(CLEAN_WAV_PATH, os.path.join(export_folder, f'{VOICE_ID}.wav'))
shutil.copy(bin_path, os.path.join(export_folder, f'{VOICE_ID}.bin'))
shutil.copy(emb_path, os.path.join(export_folder, f'{VOICE_ID}.emb'))

shutil.make_archive(ZIP_OUTPUT.replace('.zip', ''), 'zip', export_folder)
print(f"\n🎉 Archive prête au téléchargement : {ZIP_OUTPUT}")

## 8. Intégration dans NekoSpeak

### Méthode 1 : Zéro-code (Directement sur votre S25 Ultra)
1. Transférez le fichier `{VOICE_ID}.wav` sur votre téléphone.
2. Dans NekoSpeak, allez dans **Voix** → **Clonage de voix** → **« Importer un audio »**.
3. Sélectionnez le WAV : votre voix personnalisée est prête et s'exécute en temps réel !

### Méthode 2 : Téléchargeable depuis Hugging Face ou en dur
1. Uploadez `{VOICE_ID}.wav` sur votre dépôt Hugging Face.
2. Déclarez la voix dans `VoiceDefinitions.kt` (liste `POCKET_VOICES`) et `ModelRepository.kt`.